# DNX Cauchos S.R.L. — 2. Solución de Big Data en Databricks

| N° | Punto de la rúbrica | Sección del notebook |
|---|---|---|
| 1 | Identificación de los datos | 1 |
| 2 | Arquitectura | 2 |
| 3 | Procesamiento con Databricks (DataFrames de Apache Spark) | 3 |
| 4 | Análisis exploratorio (EDA) | 4 |
| 5 | Limpieza y preparación | 5 |
| 6 | Persistencia (tablas de Databricks) | 6 |
| 7 | Indicadores y visualizaciones | 7 |
| 8 | Inteligencia artificial (Genie) | 8 |

## 1. Identificación de los datos
Se reutilizan los datos del proyecto de BI: el Data Mart `dm_DNX_Cauchos_SRL` exportado desde SQL Server a 6 archivos CSV. Las dimensiones cliente, producto y vendedor se exportaron multiplicadas x400 (`CROSS JOIN`) para trabajar con mayor volumen y con duplicados reales.

| Archivo | Contenido | Filas |
|---|---|---|
| `dim_cliente.csv` | Clientes: código, nombre, tipo, DNI/RUC, teléfono, distrito | 240 000 |
| `dim_producto.csv` | Suelas: nombre, talla, tipo de suela, stock, costo, precio, categoría, material | 24 000 |
| `dim_vendedor.csv` | Vendedores: DNI, nombre, apellido, usuario, cargo | 24 000 |
| `dim_metodoPago.csv` | Métodos de pago | 4 |
| `dim_tiempo.csv` | Calendario 2025-2026 | 688 |
| `Hecho_ventas.csv` | Líneas de venta: cantidad, precio, monto, descuento, total, costo, utilidad | 2 828 |

**Problemática del negocio:** controlar los descuentos frente al tope anual de S/ 30,000 y conocer qué productos, vendedores y distritos generan más ventas en 2025 y 2026.

## 2. Arquitectura
`SQL Server (Data Mart)` → `Exportación SSMS a CSV` → `Databricks Volume` → `Apache Spark (DataFrames en PySpark)` → `Tablas Delta de Databricks` → `Dashboard de Databricks` + `Genie`


## 3. Procesamiento con Databricks — carga en DataFrames de Apache Spark

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Ubicación de los CSV (Volume) y del esquema donde se guardarán las tablas
RUTA    = "/Volumes/empresa_dnx_cauchos/dnx_cauchos_big_data/dnx_cauchos"
CATALOG = "empresa_dnx_cauchos"
SCHEMA  = "dnx_cauchos_big_data"

TOPE_DESCUENTO  = 30000.0   # tope anual de descuentos (S/)
UMBRAL_AMARILLO = 0.80      # alerta amarilla desde el 80 % del tope; roja al superarlo

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE {CATALOG}.{SCHEMA}")

display(dbutils.fs.ls(RUTA))

path,name,size,modificationTime
dbfs:/Volumes/empresa_dnx_cauchos/dnx_cauchos_big_data/dnx_cauchos/Hecho_ventas.csv,Hecho_ventas.csv,246915,1795487553000
dbfs:/Volumes/empresa_dnx_cauchos/dnx_cauchos_big_data/dnx_cauchos/dim_cliente.csv,dim_cliente.csv,20300065,1795487555000
dbfs:/Volumes/empresa_dnx_cauchos/dnx_cauchos_big_data/dnx_cauchos/dim_metodoPago.csv,dim_metodoPago.csv,198,1795487553000
dbfs:/Volumes/empresa_dnx_cauchos/dnx_cauchos_big_data/dnx_cauchos/dim_producto.csv,dim_producto.csv,2255711,1795487553000
dbfs:/Volumes/empresa_dnx_cauchos/dnx_cauchos_big_data/dnx_cauchos/dim_tiempo.csv,dim_tiempo.csv,24137,1795487553000
dbfs:/Volumes/empresa_dnx_cauchos/dnx_cauchos_big_data/dnx_cauchos/dim_vendedor.csv,dim_vendedor.csv,1893271,1795487553000


In [0]:
archivos = {f.name.lower(): f.path for f in dbutils.fs.ls(RUTA)}

def leer_csv(nombre):
    df = (spark.read.format("csv")
          .option("header", True)
          .option("inferSchema", False)      # se lee como texto para revisar el dato original
          .option("encoding", "UTF-8")
          .load(archivos[nombre]))
    # withColumnRenamed: quitar el carácter BOM que SSMS deja en la primera cabecera
    for c in df.columns:
        if c != c.replace("﻿", "").strip():
            df = df.withColumnRenamed(c, c.replace("﻿", "").strip())
    return df

df_cliente   = leer_csv("dim_cliente.csv")
df_producto  = leer_csv("dim_producto.csv")
df_vendedor  = leer_csv("dim_vendedor.csv")
df_metodo    = leer_csv("dim_metodopago.csv")
df_tiempo    = leer_csv("dim_tiempo.csv")
df_ventas    = leer_csv("hecho_ventas.csv")

dfs = {"dim_cliente": df_cliente, "dim_producto": df_producto, "dim_vendedor": df_vendedor,
       "dim_metodoPago": df_metodo, "dim_tiempo": df_tiempo, "hecho_ventas": df_ventas}

for nombre, df in dfs.items():
    print(f"{nombre:<15} {df.count():>9,} filas  {len(df.columns)} columnas")

dim_cliente       240,000 filas  6 columnas
dim_producto       24,000 filas  10 columnas
dim_vendedor       24,000 filas  7 columnas
dim_metodoPago          4 filas  4 columnas
dim_tiempo            688 filas  7 columnas
hecho_ventas        2,828 filas  15 columnas


In [0]:
df_ventas.printSchema()
display(df_ventas)
display(df_cliente)

root
 |-- id_hecho: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- id_producto: string (nullable = true)
 |-- id_vendedor: string (nullable = true)
 |-- id_metodoP: string (nullable = true)
 |-- id_tiempo: string (nullable = true)
 |-- pk_venta: string (nullable = true)
 |-- pk_detalle: string (nullable = true)
 |-- cantidad: string (nullable = true)
 |-- precio_unitario: string (nullable = true)
 |-- monto_venta: string (nullable = true)
 |-- descuento: string (nullable = true)
 |-- total_venta: string (nullable = true)
 |-- costo_total: string (nullable = true)
 |-- utilidad: string (nullable = true)



id_hecho,id_cliente,id_producto,id_vendedor,id_metodoP,id_tiempo,pk_venta,pk_detalle,cantidad,precio_unitario,monto_venta,descuento,total_venta,costo_total,utilidad
1,14,3,3,1,2,VNT-00001,DET-00001,40,18.50,740.00,.00,740.00,500.00,240.00
2,14,17,3,1,2,VNT-00001,DET-00002,25,22.00,550.00,10.00,540.00,300.00,240.00
3,127,8,18,2,3,VNT-00002,DET-00003,60,15.80,948.00,15.00,933.00,1500.00,-567.00
4,127,24,18,2,3,VNT-00002,DET-00004,30,24.50,735.00,.00,735.00,315.00,420.00
5,127,41,18,2,3,VNT-00002,DET-00005,50,19.00,950.00,.00,950.00,1225.00,-275.00
6,412,5,9,4,5,VNT-00004,DET-00007,35,26.00,910.00,.00,910.00,472.50,437.50
7,412,33,9,4,5,VNT-00004,DET-00008,45,17.50,787.50,.00,787.50,630.00,157.50
8,533,19,55,1,6,VNT-00005,DET-00009,100,13.90,1390.00,30.00,1360.00,800.00,560.00
9,533,47,55,1,6,VNT-00005,DET-00010,20,29.00,580.00,.00,580.00,230.00,350.00
10,45,1,27,2,7,VNT-00006,DET-00011,50,16.50,825.00,.00,825.00,575.00,250.00


pk_cliente,nombre_cliente,tipo_cliente,dni,telefono,distrito
CLI-00001,Calzados El Inca S.A.C.,Cliente corporativo,20512345601,987654321,San Juan de Lurigancho
CLI-00002,Carlos Garcia Aliaga,Cliente normal,41234502,981234502,Ate
CLI-00003,Industrias Lima S.A.,Cliente corporativo,20523456703,982345603,El Agustino
CLI-00004,Luis Quispe Guerra,Cliente normal,42345604,null,Comas
CLI-00005,Manufacturas Trujillo E.I.R.L.,Cliente corporativo,20534567805,983456705,Los Olivos
CLI-00006,Jorge Martinez Rios,Cliente normal,43456706,984567806,Villa El Salvador
CLI-00007,Corporacion Moderna S.R.L.,Cliente corporativo,20545678907,null,Rimac
CLI-00008,Miguel Soto Rivas,Cliente normal,44567808,985678908,Puente Piedra
CLI-00009,Fabrica de Botas Peru S.A.C.,Cliente corporativo,20556789009,986789009,San Martin de Porres
CLI-00010,Jose Mendoza Luna,Cliente normal,45678910,987890110,La Victoria


## 4. Análisis exploratorio (EDA)
### 4.1 Volumen y registros duplicados

In [0]:
resumen = []
for nombre, df in dfs.items():
    total = df.count()
    distintos = df.distinct().count()
    resumen.append((nombre, total, distintos, total - distintos))

display(spark.createDataFrame(resumen, "tabla string, filas long, filas_distintas long, duplicados long"))

tabla,filas,filas_distintas,duplicados
dim_cliente,240000,600,239400
dim_producto,24000,60,23940
dim_vendedor,24000,60,23940
dim_metodoPago,4,4,0
dim_tiempo,688,688,0
hecho_ventas,2828,2828,0


### 4.2 Estadística descriptiva (`describe`)

In [0]:
MEDIDAS = ["cantidad", "precio_unitario", "monto_venta", "descuento", "total_venta", "costo_total", "utilidad"]
ventas_num = df_ventas.select([F.col(c).cast("double").alias(c) for c in MEDIDAS])
display(ventas_num.describe())

summary,cantidad,precio_unitario,monto_venta,descuento,total_venta,costo_total,utilidad
count,2828,2828,2828,2828,2828,2828,2828
mean,90.77263083451203,20.18299151343718,1728.0636492220651,24.883309759547384,1703.1803394625176,1597.682991513437,105.49734794908062
stddev,30.68655969226496,4.799793353196683,422.9127796037313,14.783683634714883,412.3971628045434,907.0974296751162,832.3707500397844
min,20.0,12.5,550.0,0.0,540.0,230.0,-2660.0
max,185.0,31.0,3080.0,70.0,3030.0,4995.0,1702.5


In [0]:
display(df_producto.select(F.col("costo_fabricacion").cast("double"), F.col("precio").cast("double"),
                           F.col("stock").cast("int")).describe())

summary,costo_fabricacion,precio,stock
count,24000,24000,24000
mean,18.133333333333333,34.71666666666667,455.0
stddev,8.126190600734077,14.760718156836957,87.89381029635692
min,8.0,16.0,300
max,35.0,67.0,600


### 4.3 Valores nulos o vacíos por columna

In [0]:
nulos = []
for nombre, df in dfs.items():
    fila = df.select([
        F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
        for c in df.columns]).first().asDict()
    nulos += [(nombre, c, int(v)) for c, v in fila.items() if v > 0]

display(spark.createDataFrame(nulos, "tabla string, columna string, nulos_o_vacios long"))

tabla,columna,nulos_o_vacios
dim_cliente,telefono,53600


### 4.4 Distritos mal escritos (espacios y minúsculas)

In [0]:
display(df_cliente
        .filter((F.col("distrito") != F.trim("distrito")) |
                (F.substring(F.trim("distrito"), 1, 1) == F.lower(F.substring(F.trim("distrito"), 1, 1))))
        .groupBy(F.concat(F.lit("["), F.col("distrito"), F.lit("]")).alias("distrito_original"))
        .count()
        .orderBy("distrito_original"))

distrito_original,count
[ La Victoria ],800
[ Los Olivos ],1200
[ San Juan de Lurigancho ],800
[ comas ],1200
[ate],800


### 4.5 Clientes con el mismo DNI/RUC y distinto código

In [0]:
display(df_cliente.select("pk_cliente", "nombre_cliente", "dni").distinct()
        .groupBy("dni", "nombre_cliente")
        .agg(F.count("*").alias("codigos"), F.array_sort(F.collect_list("pk_cliente")).alias("lista_codigos"))
        .filter("codigos > 1")
        .orderBy("dni"))

dni,nombre_cliente,codigos,lista_codigos
20512345601,Calzados El Inca S.A.C.,2,"List(CLI-00001, CLI-00596)"
20523456703,Industrias Lima S.A.,2,"List(CLI-00003, CLI-00598)"
20534567805,Manufacturas Trujillo E.I.R.L.,2,"List(CLI-00005, CLI-00600)"
41234502,Carlos Garcia Aliaga,2,"List(CLI-00002, CLI-00597)"
42345604,Luis Quispe Guerra,2,"List(CLI-00004, CLI-00599)"


### 4.6 Comportamiento de las ventas

In [0]:
# Ventas y descuento por año (id_tiempo 1 = 2025-01-02, primer día del calendario)
ventas_tiempo = df_ventas.withColumn(
    "fecha", F.date_add(F.lit("2025-01-02").cast("date"), F.col("id_tiempo").cast("int") - 1))

display(ventas_tiempo.groupBy(F.year("fecha").alias("anio"))
        .agg(F.countDistinct("pk_venta").alias("ventas"),
             F.round(F.sum(F.col("total_venta").cast("double")), 2).alias("ventas_netas"),
             F.round(F.sum(F.col("descuento").cast("double")), 2).alias("descuento"),
             F.round(F.sum(F.col("utilidad").cast("double")), 2).alias("utilidad"))
        .orderBy("anio"))

anio,ventas,ventas_netas,descuento,utilidad
2025,687,2068608.0,26610.0,122128.0
2026,719,2747986.0,43760.0,176218.5


In [0]:
display(ventas_tiempo.groupBy(F.date_format("fecha", "yyyy-MM").alias("mes"))
        .agg(F.round(F.sum(F.col("total_venta").cast("double")), 2).alias("ventas_netas"))
        .orderBy("mes"))

mes,ventas_netas
2025-01,140872.0
2025-02,131369.5
2025-03,155406.5
2025-04,164082.5
2025-05,167348.0
2025-06,174769.0
2025-07,198918.5
2025-08,192383.0
2025-09,183869.0
2025-10,175138.0


### 4.7 Valores atípicos (método IQR) y ventas con pérdida

In [0]:
cuartiles = ventas_num.agg(*[F.percentile_approx(c, [0.25, 0.75]).alias(c) for c in MEDIDAS]).first()
atipicos = []
for c in MEDIDAS:
    q1, q3 = cuartiles[c]
    li, ls = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    atipicos.append((c, float(q1), float(q3), float(li), float(ls),
                     ventas_num.filter((F.col(c) < li) | (F.col(c) > ls)).count()))
display(spark.createDataFrame(atipicos, "medida string, q1 double, q3 double, limite_inferior double, limite_superior double, atipicos long"))

display(ventas_num.filter(F.col("utilidad") < 0)
        .agg(F.count("*").alias("lineas_con_perdida"),
             F.round(F.sum("utilidad"), 2).alias("perdida_total")))

medida,q1,q3,limite_inferior,limite_superior,atipicos
cantidad,70.0,110.0,10.0,170.0,16
precio_unitario,16.0,23.4,4.900000000000002,34.5,0
monto_venta,1440.0,2012.5,581.25,2871.25,11
descuento,15.0,35.0,-15.0,65.0,5
total_venta,1425.0,1970.0,607.5,2787.5,17
costo_total,900.0,2100.0,-900.0,3900.0,70
utilidad,-520.0,702.5,-2353.75,2536.25,31


lineas_con_perdida,perdida_total
968,-850513.5


### 4.8 Relación entre la tabla de ventas y las dimensiones
Se comprueba si el `id_producto` de las ventas apunta al producto correcto: el costo unitario de la venta (`costo_total / cantidad`) debe ser igual al `costo_fabricacion` del producto.

In [0]:
prod = (df_producto.select(F.col("id_producto").cast("int").alias("id_producto"), "pk_producto",
                           F.col("costo_fabricacion").cast("double").alias("costo_fab"))
        .distinct())
vta = df_ventas.select(F.col("id_producto").cast("int").alias("id_producto"),
                       (F.col("costo_total").cast("double") / F.col("cantidad").cast("double")).alias("costo_unit"))

def coincidencia(df):
    return df.agg(F.count("*").alias("lineas"),
                  F.sum(F.when(F.col("pk_producto").isNull(), 1).otherwise(0)).alias("sin_producto"),
                  F.round(100 * F.avg(F.when(F.abs(F.col("costo_unit") - F.col("costo_fab")) < 0.001, 1.0)
                                        .otherwise(0.0)), 2).alias("pct_costo_coincide"))

print("A) Uniendo por id_producto:")
display(coincidencia(vta.join(prod, "id_producto", "left")))

print("B) Uniendo por el número del código del producto (PRO-00003 -> 3):")
prod_codigo = prod.drop("id_producto").withColumn("id_producto", F.regexp_extract("pk_producto", r"(\d+)$", 1).cast("int"))
display(coincidencia(vta.join(prod_codigo, "id_producto", "left")))

# Rango de claves: las dimensiones empiezan en 2 y las ventas en 1
rangos = []
for nombre, df, col in [("dim_producto", df_producto, "id_producto"), ("dim_vendedor", df_vendedor, "id_vendedor"),
                        ("dim_metodoPago", df_metodo, "id_metodoP"), ("dim_tiempo", df_tiempo, "id_tiempo")]:
    d = df.agg(F.min(F.col(col).cast("int")), F.max(F.col(col).cast("int"))).first()
    v = df_ventas.agg(F.min(F.col(col).cast("int")), F.max(F.col(col).cast("int"))).first()
    rangos.append((nombre, d[0], d[1], v[0], v[1]))
display(spark.createDataFrame(rangos, "dimension string, min_dimension int, max_dimension int, min_ventas int, max_ventas int"))

A) Uniendo por id_producto:


lineas,sin_producto,pct_costo_coincide
2828,33,0.0


B) Uniendo por el número del código del producto (PRO-00003 -> 3):


lineas,sin_producto,pct_costo_coincide
2828,0,100.0


dimension,min_dimension,max_dimension,min_ventas,max_ventas
dim_producto,2,61,1,50
dim_vendedor,2,61,1,60
dim_metodoPago,2,5,1,4
dim_tiempo,2,689,1,688


**Resultado del EDA:**
* Las dimensiones cliente, producto y vendedor están repetidas 400 veces.
* `telefono` tiene valores vacíos y `distrito` tiene espacios y minúsculas.
* Hay clientes registrados dos veces con el mismo DNI/RUC.
* Las claves `id_*` de las dimensiones están desplazadas en +1 respecto a las ventas: unir por `id_producto` asigna el producto equivocado. Uniendo por el código del producto, el costo coincide al 100 %.

## 5. Limpieza y preparación
### 5.1 Clientes

In [0]:
cliente = (df_cliente
    .select("pk_cliente", "nombre_cliente", "tipo_cliente", "dni", "telefono", "distrito")   # select
    .dropDuplicates()                                                                        # quita las 400 repeticiones
    .withColumn("pk_cliente", F.upper(F.trim("pk_cliente")))
    .withColumn("nombre_cliente", F.trim("nombre_cliente"))
    .withColumn("dni", F.trim("dni"))
    .withColumn("telefono", F.when(F.trim("telefono") == "", F.lit(None)).otherwise(F.trim("telefono")))
    .withColumn("distrito", F.regexp_replace(F.trim("distrito"), r"\s+", " ")))

# Distrito: se toma la escritura más frecuente de cada distrito (groupBy) -> 'ate' pasa a 'Ate'
w = Window.partitionBy("clave").orderBy(F.desc("n"))
distrito_ok = (cliente.withColumn("clave", F.lower("distrito"))
               .groupBy("clave", "distrito").agg(F.count("*").alias("n"))
               .withColumn("r", F.row_number().over(w)).filter("r = 1")
               .select("clave", F.col("distrito").alias("distrito_ok")))

cliente = (cliente.withColumn("clave", F.lower("distrito"))
           .join(distrito_ok, "clave")
           .drop("clave", "distrito")
           .withColumnRenamed("distrito_ok", "distrito")                                    # withColumnRenamed
           .withColumn("id_cliente", F.regexp_extract("pk_cliente", r"(\d+)$", 1).cast("int")))

# Clientes duplicados por DNI: se conserva el código más antiguo
w_dni = Window.partitionBy("dni").orderBy("id_cliente")
cliente = cliente.withColumn("id_cliente_final", F.first("id_cliente").over(w_dni))
duplicados = cliente.filter("id_cliente <> id_cliente_final").select("id_cliente", "id_cliente_final")
display(duplicados)

cliente = (cliente.filter("id_cliente = id_cliente_final")                                    # filter
           .drop("id_cliente_final")
           .na.fill({"telefono": "SIN REGISTRO"})                                               # na
           .select("id_cliente", "pk_cliente", "nombre_cliente", "tipo_cliente", "dni", "telefono", "distrito"))

print("Clientes:", df_cliente.count(), "->", cliente.count())
display(cliente.groupBy("distrito").count().orderBy("distrito"))

id_cliente,id_cliente_final
596,1
598,3
600,5
597,2
599,4


Clientes: 240000 -> 595


distrito,count
Ate,30
Brena,30
Carabayllo,30
Cercado de Lima,29
Chorrillos,29
Comas,30
El Agustino,30
Independencia,30
Jesus Maria,29
La Victoria,30


### 5.2 Productos, vendedores, métodos de pago y tiempo
La clave `id_*` se reconstruye a partir del código de negocio (`PRO-00003` → 3) para que coincida con la tabla de ventas.

In [0]:
producto = (df_producto
    .select("pk_producto", "nombre_producto", "tipo_suela", "talla", "stock", "costo_fabricacion", "precio", "categoria", "material")
    .dropDuplicates()
    .withColumn("id_producto", F.regexp_extract("pk_producto", r"(\d+)$", 1).cast("int"))
    .withColumn("talla", F.col("talla").cast("int"))
    .withColumn("stock", F.col("stock").cast("int"))
    .withColumn("costo_fabricacion", F.col("costo_fabricacion").cast("decimal(10,2)"))
    .withColumn("precio", F.col("precio").cast("decimal(10,2)"))
    .filter((F.col("precio") > 0) & (F.col("costo_fabricacion") > 0)))

vendedor = (df_vendedor
    .select("pk_vendedor", "dni", "nombre_vendedor", "apellido", "usuario", "cargo")
    .dropDuplicates()
    .withColumn("id_vendedor", F.regexp_extract("pk_vendedor", r"(\d+)$", 1).cast("int"))
    .withColumn("vendedor", F.concat_ws(" ", F.trim("nombre_vendedor"), F.trim("apellido"))))

metodo_pago = (df_metodo
    .select("pk_metodoP", "nombre_metodo", "tipo_pago")
    .dropDuplicates()
    .withColumnRenamed("pk_metodoP", "pk_metodo_pago")
    .withColumn("id_metodo_pago", F.regexp_extract("pk_metodo_pago", r"(\d+)$", 1).cast("int")))

tiempo = (df_tiempo
    .select("fecha", "nombre_mes")
    .dropDuplicates()
    .withColumn("fecha", F.to_date("fecha"))
    .withColumn("id_tiempo", F.datediff("fecha", F.lit("2025-01-02").cast("date")) + 1)   # 1 = primer día
    .withColumn("dia", F.dayofmonth("fecha"))
    .withColumn("mes", F.month("fecha"))
    .withColumn("trimestre", F.quarter("fecha"))
    .withColumn("anio", F.year("fecha")))

for n, df in [("producto", producto), ("vendedor", vendedor), ("metodo_pago", metodo_pago), ("tiempo", tiempo)]:
    print(f"{n:<12} {df.count():>5} filas")

producto        60 filas
vendedor        60 filas
metodo_pago      4 filas
tiempo         688 filas


### 5.3 Ventas

In [0]:
ventas = (df_ventas
    .select("pk_venta", "pk_detalle", "id_cliente", "id_producto", "id_vendedor", "id_metodoP", "id_tiempo", *MEDIDAS)
    .withColumnRenamed("id_metodoP", "id_metodo_pago")
    .dropDuplicates(["pk_detalle"]))

for c in ["id_cliente", "id_producto", "id_vendedor", "id_metodo_pago", "id_tiempo", "cantidad"]:
    ventas = ventas.withColumn(c, F.col(c).cast("int"))
for c in ["precio_unitario", "monto_venta", "descuento", "total_venta", "costo_total", "utilidad"]:
    ventas = ventas.withColumn(c, F.col(c).cast("decimal(12,2)"))

# Ventas de clientes duplicados -> cliente que se conserva
ventas = (ventas.join(duplicados, "id_cliente", "left")
                .withColumn("id_cliente", F.coalesce("id_cliente_final", "id_cliente"))
                .drop("id_cliente_final"))

# Reglas de calidad
antes = ventas.count()
ventas = (ventas.na.drop(subset=["pk_venta", "pk_detalle", "cantidad", "total_venta"])
                .filter((F.col("cantidad") > 0) & (F.col("precio_unitario") > 0))
                .filter((F.col("descuento") >= 0) & (F.col("descuento") <= F.col("monto_venta")))
                .filter(F.abs(F.col("total_venta") - (F.col("monto_venta") - F.col("descuento"))) <= 0.01))
# Evidencia de los cambios
print("Líneas de clientes duplicados reasignadas:",
      df_ventas.filter(F.col("id_cliente").cast("int").between(596, 600)).count())
print("Líneas que aún apuntan a CLI-00596 a 600:",
      ventas.filter(F.col("id_cliente").between(596, 600)).count())


Ventas: 2828 -> 2828
Líneas de clientes duplicados reasignadas: 82
Líneas que aún apuntan a CLI-00596 a 600: 0


In [0]:
# Comprobación: todas las ventas encuentran su cliente, producto, vendedor, método y fecha
for nombre, df, k in [("cliente", cliente, "id_cliente"), ("producto", producto, "id_producto"),
                      ("vendedor", vendedor, "id_vendedor"), ("metodo_pago", metodo_pago, "id_metodo_pago"),
                      ("tiempo", tiempo, "id_tiempo")]:
    print(f"{nombre:<12} ventas sin relación: {ventas.join(df.select(k), k, 'left_anti').count()}")

cliente      ventas sin relación: 0
producto     ventas sin relación: 0
vendedor     ventas sin relación: 0
metodo_pago  ventas sin relación: 0
tiempo       ventas sin relación: 0


In [0]:
# Resumen antes / después de la limpieza
display(spark.createDataFrame([
    ("cliente",     df_cliente.count(),  cliente.count()),
    ("producto",    df_producto.count(), producto.count()),
    ("vendedor",    df_vendedor.count(), vendedor.count()),
    ("metodo_pago", df_metodo.count(),   metodo_pago.count()),
    ("tiempo",      df_tiempo.count(),   tiempo.count()),
    ("ventas",      df_ventas.count(),   ventas.count()),
], "tabla string, filas_antes long, filas_despues long"))

tabla,filas_antes,filas_despues
cliente,240000,595
producto,24000,60
vendedor,24000,60
metodo_pago,4,4
tiempo,688,688
ventas,2828,2828


## 6. Persistencia — tablas de Databricks

In [0]:
def guardar(df, tabla):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.{tabla}")
    print(f"✔ {tabla}")

guardar(cliente, "cliente")
guardar(producto, "producto")
guardar(vendedor, "vendedor")
guardar(metodo_pago, "metodo_pago")
guardar(tiempo, "tiempo")
guardar(ventas, "ventas")

# Tabla de ventas con los datos de cada dimensión (para el dashboard y Genie)
ventas_detalle = (spark.table("ventas")
    .join(spark.table("tiempo").select("id_tiempo", "fecha", "anio", "mes", "nombre_mes"), "id_tiempo")
    .join(spark.table("cliente").select("id_cliente", "nombre_cliente", "tipo_cliente", "distrito"), "id_cliente")
    .join(spark.table("producto").select("id_producto", "nombre_producto", "talla", "categoria", "material"), "id_producto")
    .join(spark.table("vendedor").select("id_vendedor", "vendedor", "cargo"), "id_vendedor")
    .join(spark.table("metodo_pago").select("id_metodo_pago", "nombre_metodo"), "id_metodo_pago")
    .select("pk_venta", "pk_detalle", "fecha", "anio", "mes", "nombre_mes", "nombre_cliente", "tipo_cliente",
            "distrito", "nombre_producto", "talla", "categoria", "material", "vendedor", "cargo", "nombre_metodo",
            "cantidad", "precio_unitario", "monto_venta", "descuento", "total_venta", "costo_total", "utilidad"))
guardar(ventas_detalle, "ventas_detalle")

✔ cliente
✔ producto
✔ vendedor
✔ metodo_pago
✔ tiempo
✔ ventas
✔ ventas_detalle


In [0]:
%sql
SHOW TABLES IN empresa_dnx_cauchos.dnx_cauchos_big_data

database,tableName,isTemporary
dnx_cauchos_big_data,cliente,false
dnx_cauchos_big_data,kpi_anual,false
dnx_cauchos_big_data,kpi_descuento_anual,false
dnx_cauchos_big_data,kpi_descuento_mensual,false
dnx_cauchos_big_data,kpi_distritos,false
dnx_cauchos_big_data,kpi_productos,false
dnx_cauchos_big_data,kpi_vendedores,false
dnx_cauchos_big_data,metodo_pago,false
dnx_cauchos_big_data,producto,false
dnx_cauchos_big_data,tiempo,false


## 7. Indicadores y visualizaciones
En cada resultado se puede pulsar **+ → Visualization** para verlo como gráfico. Las tablas de indicadores se guardan para usarlas en el dashboard de Databricks.
### 7.1 Indicadores generales por año

In [0]:
kpi_anual = (spark.table("ventas_detalle").groupBy("anio")
    .agg(F.countDistinct("pk_venta").alias("ventas"),
         F.sum("cantidad").alias("unidades"),
         F.sum("total_venta").alias("ventas_netas"),
         F.sum("descuento").alias("descuento"),
         F.sum("costo_total").alias("costo"),
         F.sum("utilidad").alias("utilidad"))
    .withColumn("margen_pct", F.round(100 * F.col("utilidad") / F.col("ventas_netas"), 2))
    .orderBy("anio"))
guardar(kpi_anual, "kpi_anual")
display(spark.table("kpi_anual"))

✔ kpi_anual


anio,ventas,unidades,ventas_netas,descuento,costo,utilidad,margen_pct
2025,687,110700,2068608.00,26610.00,1946480.00,122128.00,5.90
2026,719,146005,2747986.00,43760.00,2571767.50,176218.50,6.41


### 7.2 Control de descuentos frente al tope de S/ 30,000
Alerta **VERDE** < 80 % del tope · **AMARILLA** 80 %–100 % · **ROJA** > 100 %.

In [0]:
def alerta(pct):
    return F.when(pct > 1, "ROJA").when(pct >= UMBRAL_AMARILLO, "AMARILLA").otherwise("VERDE")

kpi_descuento_anual = (spark.table("ventas_detalle").groupBy("anio")
    .agg(F.sum("descuento").cast("double").alias("descuento_total"))
    .withColumn("tope", F.lit(TOPE_DESCUENTO))
    .withColumn("saldo", F.col("tope") - F.col("descuento_total"))
    .withColumn("pct_consumo", F.round(100 * F.col("descuento_total") / F.col("tope"), 1))
    .withColumn("alerta", alerta(F.col("descuento_total") / F.col("tope")))
    .orderBy("anio"))
guardar(kpi_descuento_anual, "kpi_descuento_anual")
display(spark.table("kpi_descuento_anual"))

✔ kpi_descuento_anual


anio,descuento_total,tope,saldo,pct_consumo,alerta
2025,26610.0,30000.0,3390.0,88.7,AMARILLA
2026,43760.0,30000.0,-13760.0,145.9,ROJA


In [0]:
w_acum = Window.partitionBy("anio").orderBy("mes").rowsBetween(Window.unboundedPreceding, Window.currentRow)
kpi_descuento_mensual = (spark.table("ventas_detalle").groupBy("anio", "mes", "nombre_mes")
    .agg(F.sum("descuento").cast("double").alias("descuento_mes"))
    .withColumn("descuento_acumulado", F.sum("descuento_mes").over(w_acum))
    .withColumn("tope", F.lit(TOPE_DESCUENTO))
    .withColumn("alerta", alerta(F.col("descuento_acumulado") / F.col("tope")))
    .orderBy("anio", "mes"))
guardar(kpi_descuento_mensual, "kpi_descuento_mensual")
display(spark.table("kpi_descuento_mensual"))

✔ kpi_descuento_mensual


anio,mes,nombre_mes,descuento_mes,descuento_acumulado,tope,alerta
2025,1,Enero,1540.0,1540.0,30000.0,VERDE
2025,2,Febrero,1370.0,2910.0,30000.0,VERDE
2025,3,Marzo,1760.0,4670.0,30000.0,VERDE
2025,4,Abril,1905.0,6575.0,30000.0,VERDE
2025,5,Mayo,1955.0,8530.0,30000.0,VERDE
2025,6,Junio,2250.0,10780.0,30000.0,VERDE
2025,7,Julio,2790.0,13570.0,30000.0,VERDE
2025,8,Agosto,2900.0,16470.0,30000.0,VERDE
2025,9,Septiembre,2570.0,19040.0,30000.0,VERDE
2025,10,Octubre,2280.0,21320.0,30000.0,VERDE


### 7.3 Productos, vendedores y distritos con más ventas

In [0]:
w_rank = Window.partitionBy("anio").orderBy(F.desc("ventas_netas"))

def ranking(columnas, tabla):
    df = (spark.table("ventas_detalle").groupBy("anio", *columnas)
          .agg(F.sum("cantidad").alias("unidades"),
               F.sum("total_venta").alias("ventas_netas"),
               F.sum("descuento").alias("descuento"),
               F.sum("utilidad").alias("utilidad"))
          .withColumn("puesto", F.dense_rank().over(w_rank))
          .orderBy("anio", "puesto"))
    guardar(df, tabla)
    return spark.table(tabla)

kpi_productos  = ranking(["nombre_producto", "categoria"], "kpi_productos")
kpi_vendedores = ranking(["vendedor", "cargo"], "kpi_vendedores")
kpi_distritos  = ranking(["distrito"], "kpi_distritos")

display(kpi_productos.filter("puesto <= 10"))
display(kpi_vendedores.filter("puesto <= 10"))
display(kpi_distritos.filter("puesto <= 10"))

✔ kpi_productos
✔ kpi_vendedores
✔ kpi_distritos


anio,nombre_producto,categoria,unidades,ventas_netas,descuento,utilidad,puesto
2025,Suela Runner Pro Air,Deportivo,16375,288534.00,4630.00,79554.00,1
2025,Suela Max Armor Dieléctrica,Seguridad,12380,261552.00,2390.00,-95385.50,2
2025,Suela Skate Grip Street,Urbano,12820,259982.00,2580.00,78444.50,3
2025,Suela City Walk Flex,Casual,15980,259961.50,4455.00,80071.50,4
2025,Suela Trekking Master 4x4,Industrial,13655,258462.50,3000.00,-55237.50,5
2025,Suela Vulcano Extrema Heavy,Seguridad,16325,238972.00,4890.00,-200623.00,6
2025,Suela Mocasín Confort Soft,Casual,8770,235535.00,1390.00,142082.50,7
2025,Suela Oxford Classic Dress,Formal,12185,197936.50,2945.00,88624.00,8
2025,Suela Botín Minero Titan,Seguridad,2210,67672.50,330.00,4597.50,9
2026,Suela City Walk Flex,Casual,22700,371355.50,7370.00,115688.00,1


anio,vendedor,cargo,unidades,ventas_netas,descuento,utilidad,puesto
2025,Eduardo Arce Romero,Ejecutivo de Cuentas Clave,2555,48061.50,630.00,424.00,1
2025,Jorge Paredes Davila,Atencion al Cliente y Ventas,2400,48048.50,540.00,4428.50,2
2025,Silvia Valdivia Cordova,Ejecutivo de Cuentas Clave,2305,47506.00,515.00,9383.50,3
2025,Hector Fuentes Rivera,Atencion al Cliente y Ventas,2555,46975.00,635.00,-45.00,4
2025,Lucia Solis Gamboa,Ejecutivo de Cuentas Clave,2335,46044.50,505.00,8402.00,5
2025,Carmen Mejia Guzman,Ejecutivo de Cuentas Clave,2370,45567.00,525.00,3217.00,6
2025,Manuel Vento Lara,Asesor Comercial,2310,44977.50,490.00,-3350.00,7
2025,Gabriela Chavez Campos,Asesor Comercial,2300,44690.00,545.00,6017.50,8
2025,Patricia Campos Cruz,Ejecutivo de Cuentas Clave,2265,44519.50,515.00,8354.50,9
2025,Marco Sosa Jimenez,Asesor Comercial Mayorista,2215,42991.00,495.00,3626.00,10


anio,distrito,unidades,ventas_netas,descuento,utilidad,puesto
2025,Comas,7950,150859.50,1960.00,12052.00,1
2025,San Juan de Lurigancho,7325,136295.50,1795.00,18610.50,2
2025,Pueblo Libre,7160,132654.00,1755.00,16529.00,3
2025,Cercado de Lima,6950,131025.50,1590.00,-3957.00,4
2025,San Martin de Porres,6405,118695.50,1555.00,14388.00,5
2025,Los Olivos,6030,113711.00,1420.00,6323.50,6
2025,San Miguel,5820,109830.00,1480.00,19200.00,7
2025,Rimac,5875,108909.00,1410.00,-5311.00,8
2025,Santa Anita,6035,108772.00,1565.00,2269.50,9
2025,Carabayllo,5860,106060.50,1485.00,1468.00,10


### 7.4 Dashboard
Crear en **Dashboards → Create dashboard** usando las tablas `kpi_anual`, `kpi_descuento_anual`, `kpi_descuento_mensual`, `kpi_productos`, `kpi_vendedores` y `kpi_distritos`, con un filtro por `anio` (2025 / 2026).

In [0]:
display(spark.table("kpi_vendedores"))

anio,vendedor,cargo,unidades,ventas_netas,descuento,utilidad,puesto
2025,Eduardo Arce Romero,Ejecutivo de Cuentas Clave,2555,48061.50,630.00,424.00,1
2025,Jorge Paredes Davila,Atencion al Cliente y Ventas,2400,48048.50,540.00,4428.50,2
2025,Silvia Valdivia Cordova,Ejecutivo de Cuentas Clave,2305,47506.00,515.00,9383.50,3
2025,Hector Fuentes Rivera,Atencion al Cliente y Ventas,2555,46975.00,635.00,-45.00,4
2025,Lucia Solis Gamboa,Ejecutivo de Cuentas Clave,2335,46044.50,505.00,8402.00,5
2025,Carmen Mejia Guzman,Ejecutivo de Cuentas Clave,2370,45567.00,525.00,3217.00,6
2025,Manuel Vento Lara,Asesor Comercial,2310,44977.50,490.00,-3350.00,7
2025,Gabriela Chavez Campos,Asesor Comercial,2300,44690.00,545.00,6017.50,8
2025,Patricia Campos Cruz,Ejecutivo de Cuentas Clave,2265,44519.50,515.00,8354.50,9
2025,Marco Sosa Jimenez,Asesor Comercial Mayorista,2215,42991.00,495.00,3626.00,10


Databricks visualization. Run in Databricks to view.

In [0]:
display(spark.table("kpi_productos"))

anio,nombre_producto,categoria,unidades,ventas_netas,descuento,utilidad,puesto
2025,Suela Runner Pro Air,Deportivo,16375,288534.00,4630.00,79554.00,1
2025,Suela Max Armor Dieléctrica,Seguridad,12380,261552.00,2390.00,-95385.50,2
2025,Suela Skate Grip Street,Urbano,12820,259982.00,2580.00,78444.50,3
2025,Suela City Walk Flex,Casual,15980,259961.50,4455.00,80071.50,4
2025,Suela Trekking Master 4x4,Industrial,13655,258462.50,3000.00,-55237.50,5
2025,Suela Vulcano Extrema Heavy,Seguridad,16325,238972.00,4890.00,-200623.00,6
2025,Suela Mocasín Confort Soft,Casual,8770,235535.00,1390.00,142082.50,7
2025,Suela Oxford Classic Dress,Formal,12185,197936.50,2945.00,88624.00,8
2025,Suela Botín Minero Titan,Seguridad,2210,67672.50,330.00,4597.50,9
2026,Suela City Walk Flex,Casual,22700,371355.50,7370.00,115688.00,1


Databricks visualization. Run in Databricks to view.

In [0]:
display(spark.table("kpi_distritos"))

anio,distrito,unidades,ventas_netas,descuento,utilidad,puesto
2025,Comas,7950,150859.50,1960.00,12052.00,1
2025,San Juan de Lurigancho,7325,136295.50,1795.00,18610.50,2
2025,Pueblo Libre,7160,132654.00,1755.00,16529.00,3
2025,Cercado de Lima,6950,131025.50,1590.00,-3957.00,4
2025,San Martin de Porres,6405,118695.50,1555.00,14388.00,5
2025,Los Olivos,6030,113711.00,1420.00,6323.50,6
2025,San Miguel,5820,109830.00,1480.00,19200.00,7
2025,Rimac,5875,108909.00,1410.00,-5311.00,8
2025,Santa Anita,6035,108772.00,1565.00,2269.50,9
2025,Carabayllo,5860,106060.50,1485.00,1468.00,10


Databricks visualization. Run in Databricks to view.

In [0]:
display(spark.table("kpi_descuento_mensual"))

anio,mes,nombre_mes,descuento_mes,descuento_acumulado,tope,alerta
2025,1,Enero,1540.0,1540.0,30000.0,VERDE
2025,2,Febrero,1370.0,2910.0,30000.0,VERDE
2025,3,Marzo,1760.0,4670.0,30000.0,VERDE
2025,4,Abril,1905.0,6575.0,30000.0,VERDE
2025,5,Mayo,1955.0,8530.0,30000.0,VERDE
2025,6,Junio,2250.0,10780.0,30000.0,VERDE
2025,7,Julio,2790.0,13570.0,30000.0,VERDE
2025,8,Agosto,2900.0,16470.0,30000.0,VERDE
2025,9,Septiembre,2570.0,19040.0,30000.0,VERDE
2025,10,Octubre,2280.0,21320.0,30000.0,VERDE


Databricks visualization. Run in Databricks to view.

In [0]:
display(spark.table("kpi_descuento_anual"))

anio,descuento_total,tope,saldo,pct_consumo,alerta
2025,26610.0,30000.0,3390.0,88.7,AMARILLA
2026,43760.0,30000.0,-13760.0,145.9,ROJA


Databricks visualization. Run in Databricks to view.

In [0]:
display(spark.table("kpi_anual"))

anio,ventas,unidades,ventas_netas,descuento,costo,utilidad,margen_pct
2025,687,110700,2068608.00,26610.00,1946480.00,122128.00,5.90
2026,719,146005,2747986.00,43760.00,2571767.50,176218.50,6.41


Databricks visualization. Run in Databricks to view.

## 8. Inteligencia artificial — Databricks Genie
Crear un espacio en **Genie → New** con las tablas `ventas_detalle`, `kpi_anual`, `kpi_descuento_anual`, `kpi_descuento_mensual`, `kpi_productos`, `kpi_vendedores` y `kpi_distritos`, y hacerle estas preguntas:
* ¿Cuánto descuento se otorgó en 2026 y qué porcentaje del tope de S/ 30,000 representa?
* ¿En qué mes de 2026 el descuento acumulado superó el tope?
* ¿Cuáles son los 5 productos con más ventas netas en 2025?
* ¿Qué vendedor tuvo más ventas en 2026?
* ¿Qué distritos tienen más ventas?